In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm

%matplotlib inline

In [ ]:
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import DataLoader
from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor, Compose, Normalize

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans

import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

In [ ]:
!pip install catboost

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from tqdm import tqdm
tqdm.pandas()

from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold,KFold

from lightgbm import LGBMClassifier
from xgboost import  XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import  RandomForestClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import f1_score


import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
pd.options.mode.chained_assignment = None  # default='warn'
#pd.set_option('display.float_format', lambda x: '%.3f' % x)
plt.rcParams["figure.figsize"] = (12, 8)
pd.set_option('display.max_columns', None)

In [ ]:
import kagglehub

# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
df = pd.read_csv('/kaggle/input/q3-ka-ai-2026/Q3_data.csv')


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:Display dataset information using info()
df.info()

In [ ]:
# Task 4: Write your code here:Show statistical description using describe()
df.describe()

In [ ]:
# Task 1: Write your code here:
# 2. Do we have missing values?
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

In [ ]:
# Analyze missing values
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data.head()

In [ ]:

drop_too_missing_data = missing_data[missing_data['Missing_Percentage'] > 50].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
drop_too_missing_data.head()


list(drop_too_missing_data['Column'].values)

In [ ]:
df_clean=df.copy()

print(f"Before: {df_clean.shape}")
df_clean = df_clean.drop(columns=list(drop_too_missing_data['Column'].values))
print(f"After dropping missing : {df_clean.shape}")
# Fill missing values with mean for each column
df_clean = df_clean.fillna(df_clean.mean())


In [ ]:
check_missing_values(df_clean)

In [ ]:
# Task 2: Write your code here:
# 4. Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# Task 3: Write your code here:
categorical_cols = df_clean.select_dtypes(include=["object"]).columns
print("Categorical Columns:", list(categorical_cols))


In [ ]:
# Task 4: Write your code here:Apply feature scaling to numerical features (Use StandardScaler)

from sklearn.preprocessing import StandardScaler

numerical_cols = df_clean.select_dtypes(include=["int64", "float64"]).columns.drop("Target")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df_clean[numerical_cols] = scaler.fit_transform(df_clean[numerical_cols])
df_clean.head()




In [ ]:
# Task 5: Write your code here:Check for target imbalance and state if it is imbalanced or not
# 1. Is the target imbalanced?
# Target distribution


# 1. Is the target imbalanced?
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df_clean, "Target")

In [ ]:
# Task 1: Write your code here:

X = df_clean.drop("Target",axis=1).copy()
y = df_clean['Target'].copy()
X.shape
y.shape

In [ ]:
# Task 2,3,4,5: Write your code here:


# Define classification model
model =CatBoostClassifier(verbose=0)

n_splits=5
scores_f1 = []
# Stratified 5-Fold Cross-Validation
# skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)




    # Stratified 5-Fold Cross-Validation
skf = KFold(n_splits=5)
for train_index, test_index in skf.split(X, y):
        # Split data into training and testing sets
        X_Train, X_Test = X.loc[train_index, :], X.loc[test_index, :]
        y_Train, y_Test = y.iloc[train_index], y.iloc[test_index]
        # Train the model
        model.fit(X_Train, y_Train)
        # Predict on the test set
        y_pred = model.predict(X_Test)

        # Calculate metrics
        scores_f1.append(f1_score(y_Test, y_pred, average='weighted'))

# for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
#   print(f"\nFold {fold_idx + 1}/{n_splits}")

#   # 1. Split data
#   X_Train, X_Test = X.loc[train_index, :], X.loc[test_index, :]
#   y_Train, y_Test = y.iloc[train_index], y.iloc[test_index]

#   # 2. Train  the model
#   model.fit(X_Train, y_Train)
#   # Predict on the test set
#   y_pred = model.predict(X_Test)

#   # Calculate metric
#   f1 = f1_score(y_test, y_pred,zero_division=0)

#   scores_f1.append(f1)





print(f"F1-Score: {np.mean(scores_f1):.4f}")

In [ ]:
# Task 1: Write your code here:
# Task 1: Write your code here:
# Plot feature importance
importance = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
})
importance = importance.sort_values('importance', ascending=True).tail(10)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'])
plt.title('Top 10 Feature Importance')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: